# 1편 카페 Koi의 장부를 연다
**쉽게 따라 해 보면서 배우는 파이썬**

위에서부터 한 셀씩 `Shift + Enter`. 설명은 사이트의 1편 페이지에 있다.

## 1-1 판매 기록 읽고 모양 보기

In [ ]:
# 1편 실습 파일 묶음(zip)을 올리고 푼다
from google.colab import files
import zipfile, os
up = files.upload()
zipfile.ZipFile(list(up)[0]).extractall()
print(sorted(f for f in os.listdir() if f.startswith("koi_")))

In [ ]:
# 판매 기록(CSV)을 표로 읽는다
import pandas as pd
enc = "utf-8-sig"
sales = pd.read_csv("koi_sales.csv", encoding=enc, parse_dates=["date"])
sales_raw = sales.copy()          # 나중에 비교할 원본을 따로 둔다
print("rows x cols:", sales.shape)
print(sales.head(3))

## 1-2 자료형과 빈칸부터

In [ ]:
# 열마다 자료형, 빈칸(NaN) 수, 메뉴별 줄 수
print(sales.dtypes)
print(sales.isna().sum())
print(sales["menu"].value_counts())

## 1-3 지저분한 원가표 엑셀 날것으로 보기

In [ ]:
# header=None: 첫 줄을 열 이름으로 쓰지 말고 있는 그대로 보여 달라
raw = pd.read_excel("koi_cost.xlsx", header=None)
print(raw)

## 1-4 청소 ① 제목 줄, 빈 행, 열 이름 공백

In [ ]:
# 위 두 줄(제목)을 건너뛰고 읽는다
cost = pd.read_excel("koi_cost.xlsx", skiprows=2)
print("before:", list(cost.columns))
cost = cost.dropna(how="all")              # 칸이 전부 빈 줄만 지운다
cost.columns = cost.columns.str.strip()    # 열 이름 앞뒤 공백 지우기
print("after: ", list(cost.columns))
print(cost)

## 1-5 청소 ② "₩4,500", 날짜, "-"

In [ ]:
# ₩ 와 콤마를 지우고 숫자로 바꾼다
for col in ["price", "cost"]:
    cost[col] = pd.to_numeric(cost[col].str.replace("₩", "").str.replace(",", ""), errors="coerce")
# 날짜 글자를 날짜형으로. '-' 처럼 날짜가 아닌 것은 NaT(빈 날짜)가 된다
cost["updated"] = pd.to_datetime(cost["updated"], errors="coerce")
print(cost)
print(cost.dtypes)

## 1-6 배달앱 주문 JSON 읽기

In [ ]:
# JSON 은 사전(dict) 모양으로 읽힌다. 표는 "orders" 안에 있다
import json
with open("koi_delivery.json", encoding="utf-8") as f:
    reply = json.load(f)
print(reply["status"], reply["month"], reply["count"])
delivery = pd.DataFrame(reply["orders"])
delivery["amount"] = delivery["qty"] * delivery["price"]
print(delivery.head())
print("orders:", len(delivery), "/ total:", delivery["amount"].sum())

## 1-7 열 고르고 조건 거르기

In [ ]:
# 열 하나, 여러 열
print(sales["menu"].head(3))
print(sales[["date", "menu", "qty"]].head(3))
# 조건: 7월 줄만
july = sales[sales["date"].dt.month == 7]
print("July rows:", len(july))
# 조건 두 개: 아이스티를 3개 이상 산 줄
big_tea = sales[(sales["menu"] == "Iced Tea") & (sales["qty"] >= 3)]
print("Iced Tea x3+:", len(big_tea))
# 같은 조건을 글자로
print(len(sales.query("menu == 'Iced Tea' and qty >= 3")))

## 1-8 새 열: 음료/디저트 분류와 순매출

In [ ]:
# 메뉴 → 분류 사전
kind = {"Americano": "Drink", "Latte": "Drink", "Iced Tea": "Drink",
        "Cheesecake": "Dessert", "Croissant": "Dessert"}
sales["category"] = sales["menu"].map(kind)
# 순매출 = 수량 × 단가 − 할인
sales["net"] = sales["qty"] * sales["price"] - sales["discount"]
print(sales[["menu", "category", "qty", "price", "discount", "net"]].head())
print("unmapped:", sales["category"].isna().sum())
print("total net:", sales["net"].sum())

## 1-9 map 으로 원가 붙이기, 그런데 두 메뉴가 빈다

In [ ]:
# 원가표의 두 열로 사전을 만든다 (엑셀의 VLOOKUP 자리)
unit = dict(zip(cost["menu"], cost["cost"]))
sales["unit_cost"] = sales["menu"].map(unit)
print("missing:", sales["unit_cost"].isna().sum())
print(sales.loc[sales["unit_cost"].isna(), "menu"].value_counts())

In [ ]:
print([repr(m) for m in cost["menu"]])

## 1-10 공백 지우고 merge, 합계까지 맞추기

In [ ]:
# 이름 끝 공백을 지우고, 판매 기록 왼쪽에 원가를 붙인다
cost["menu"] = cost["menu"].str.strip()
m = sales.drop(columns="unit_cost").merge(cost[["menu", "cost"]], on="menu", how="left")
print("rows:", len(sales), "->", len(m))
print("missing cost:", m["cost"].isna().sum())
m["cogs"] = m["qty"] * m["cost"]        # 재료비 = 수량 × 한 개 원가
print(m[["menu", "qty", "net", "cost", "cogs"]].head())

## 1-11 월별 손익표와 CH-01 월별 손익 막대

In [ ]:
# 월별로 묶어 더한다
m["month"] = m["date"].dt.month
fixed = pd.read_csv("koi_fixed.csv", encoding=enc)
pl = m.groupby("month")[["net", "cogs"]].sum()
pl["fixed"] = fixed.groupby("month")["amount"].sum()
pl["profit"] = pl["net"] - pl["cogs"] - pl["fixed"]
print((pl / 10000).round(1))     # 만원 단위로 보기

In [ ]:
import matplotlib.pyplot as plt
k = pl / 1e4                      # 만원
fig, ax = plt.subplots(figsize=(8, 3.6))
x = k.index
ax.bar(x - 0.2, k["net"], width=0.4, label="Net sales", color="#028090")
ax.bar(x + 0.2, k["cogs"] + k["fixed"], width=0.4, label="Costs", color="#D98E22")
ax.plot(x, k["profit"], marker="o", color="#1C2620", label="Profit")
ax.axhline(0, color="gray", linewidth=0.8)
ax.set(title="CH-01 Cafe Koi monthly P&L (2025)", xlabel="Month", ylabel="KRW 10k", xticks=list(x), ylim=(0, 1400))
ax.legend(ncol=3, loc="upper left")
plt.show()

## 1-12 검증 세 가지, 잡아낸 것 바로잡기

In [ ]:
# 검증 세 가지
c1 = m["receipt_id"].duplicated().sum()            # ① 두 번 찍힌 영수증 줄
c2 = pl["net"].sum() - m["net"].sum()               # ② 월별 합 − 전체 합 (0 이어야)
c3 = m[["net", "cost"]].isna().sum().sum()          # ③ 계산에 쓴 열의 빈칸
print("① duplicated receipts:", c1)
print("② monthly sum - total:", c2)
print("③ missing values:", c3)

In [ ]:
print(m.loc[m["receipt_id"].duplicated(keep=False), ["receipt_id", "date", "menu", "qty", "net"]])

In [ ]:
m2 = m.drop_duplicates(subset="receipt_id")
pl2 = m2.groupby("month")[["net", "cogs"]].sum()
pl2["fixed"] = pl["fixed"]
pl2["profit"] = pl2["net"] - pl2["cogs"] - pl2["fixed"]
print("rows:", len(m), "->", len(m2))
print("profit change by month:")
print((pl2["profit"] - pl["profit"])[lambda s: s != 0])
print("duplicated now:", m2["receipt_id"].duplicated().sum())

## 1-13 CH-02 재고 폭포

In [ ]:
st = pd.read_csv("koi_stock.csv", encoding=enc)
kg = st.groupby("type")["kg"].sum().round(1)
print(kg)
calc = round(kg["opening"] + kg["in"] - kg["used"] - kg["waste"], 1)
print("calculated closing:", calc, "/ counted:", kg["count"], "/ same:", calc == kg["count"])

In [ ]:
steps  = ["Opening", "In", "Used", "Waste", "Closing"]
height = [kg["opening"], kg["in"], kg["used"], kg["waste"], calc]
bottom = [0, kg["opening"], kg["opening"] + kg["in"] - kg["used"], calc, 0]
colors = ["#1F3A5F", "#028090", "#D98E22", "#C9382A", "#1F3A5F"]
fig, ax = plt.subplots(figsize=(6.5, 3.4))
ax.bar(steps, height, bottom=bottom, color=colors)
for i, (h, b) in enumerate(zip(height, bottom)):
    sign = "-" if steps[i] in ("Used", "Waste") else ("+" if steps[i] == "In" else "")
    ax.text(i, b + h + 0.8, f"{sign}{h:.1f}", ha="center")
ax.set(title="CH-02 Cafe Koi coffee beans, April 2025 (kg)", ylabel="kg", ylim=(0, 38))
plt.show()